# Runnable 인터페이스 기초

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `RunnablePassthrough`로 입력을 그대로 전달하거나 `.assign()`으로 새 키를 추가할 수 있어요
2. `RunnableParallel`로 여러 체인을 동시에 실행하고 결과를 딕셔너리로 받을 수 있어요
3. `RunnableLambda`로 Python 함수를 체인에 삽입해 전처리·후처리를 구현할 수 있어요
4. 세 가지 Runnable을 조합해 실무형 마케팅 파이프라인을 구성할 수 있어요

## 사전 지식

- `02_Chain.ipynb`의 파이프 연산자와 `PromptTemplate` 사용법
- `03_LCEL_basic.ipynb`의 `invoke`, `batch`, `stream` 메서드
- Python 딕셔너리, 람다 표현식

## 이전 노트북 연결

`03_LCEL_basic.ipynb`에서 체인을 다양한 방식으로 실행하는 방법을 배웠어요. 이번에는 체인 내부 데이터 흐름을 세밀하게 제어하는 세 가지 핵심 Runnable을 살펴볼게요.

아래 다이어그램은 세 Runnable의 역할과 상호관계를 보여줘요.

```mermaid
flowchart TD
    IN["원본 입력<br/>{product_name, price}"]:::input

    IN --> PT["RunnablePassthrough<br/>입력 그대로 통과"]:::process
    IN --> PA["RunnablePassthrough.assign<br/>새 키 추가<br/>discount_rate, current_time"]:::process
    PA --> RP["RunnableParallel<br/>병렬 실행"]:::process
    RP --> C1["체인 1<br/>상세설명 LLM"]:::process
    RP --> C2["체인 2<br/>한줄요약 LLM"]:::process
    RP --> C3["체인 3<br/>알림톡문구 LLM"]:::process
    C1 --> RL["RunnableLambda<br/>결과 합치기"]:::process
    C2 --> RL
    C3 --> RL
    RL --> OUT["최종 출력<br/>마케팅 패키지"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

Runnable 인터페이스(Runnable Interface)는 LangChain의 핵심 추상화 개념이에요. 모든 LangChain 컴포넌트(프롬프트, 모델, 출력 파서 등)가 이 인터페이스를 구현해요.

주요 메서드:
- `invoke()`: 단일 입력을 동기 실행
- `batch()`: 여러 입력을 배치 처리
- `stream()`: 출력을 스트리밍 방식으로 반환
- `ainvoke()`, `abatch()`, `astream()`: 비동기 버전

## 1. RunnablePassthrough

`RunnablePassthrough(러너블 패스스루)`는 입력을 변경하지 않고 그대로 전달하거나, `.assign()`으로 새 키를 추가해 전달할 수 있는 Runnable이에요.

### 왜 필요한가요?

LCEL 파이프라인에서 데이터는 한 방향으로만 흘러요. 그런데 실제로는 **원본 입력을 보존하면서 LLM 호출 결과도 함께 전달**해야 하는 경우가 많아요. `RunnablePassthrough`가 이 문제를 해결해요.

### 주요 사용법

| 사용법 | 동작 |
|--------|------|
| `RunnablePassthrough()` | 입력을 변형 없이 그대로 통과시켜요 (항등 함수처럼 동작해요) |
| `RunnablePassthrough.assign(key=fn)` | 원본 입력을 유지하면서 새로운 키-값 쌍을 동적으로 추가해요 |

> **비유**: `RunnablePassthrough`는 컨베이어 벨트 위의 물건을 다음 단계로 그대로 보내는 역할이에요. `.assign()`은 물건 위에 라벨을 붙여서 보내는 역할이에요.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

### 1.1 RunnablePassthrough() - 입력 그대로 전달

가장 기본적인 사용법이에요. 입력 딕셔너리가 변형 없이 그대로 반환돼요. 파이프라인 분기점에서 원본 데이터를 유지할 때 사용해요.

In [2]:
# ---------------------------------------------------
# RunnablePassthrough(): 입력을 변형 없이 그대로 통과시키기
# ---------------------------------------------------
# RunnablePassthrough: 항등 함수처럼 동작해요 — 입력 = 출력
# 파이프라인 분기점에서 원본 데이터를 유지할 때 사용해요

passthrough = RunnablePassthrough()
result = passthrough.invoke({"question": "안녕하세요", "user": "소민호"})
result

{'question': '안녕하세요', 'user': '소민호'}

### 1.2 RunnablePassthrough.assign() - 새 키 추가

`.assign()`은 원본 입력의 모든 키를 유지하면서 새로운 키-값 쌍을 동적으로 추가해요.

**언제 사용하면 좋을까요?**
- 체인 중간에서 원본 데이터를 보존하면서 **파생 값이나 메타데이터를 추가**할 때
- 입력 데이터를 기반으로 **동적으로 새로운 정보를 계산**할 때 (예: 할인율 계산, 타임스탬프 추가)
- 여러 단계의 체인에서 **이전 단계의 모든 정보를 유지**하면서 점진적으로 데이터를 확장할 때

In [9]:
# ---------------------------------------------------
# RunnablePassthrough.assign()으로 원본 입력에 파생 값 추가하기
# ---------------------------------------------------
# ============================================================
# TODO: assign()에 새 키를 추가해 체인을 실행해봐요
# 힌트: lambda x: 형태로 원본 입력 딕셔너리(x)에서 값을 꺼내 계산해요
# 예상 결과: product_name, price, discount_rate, current_time이 모두 포함된 응답이 나와요
# ============================================================

from datetime import datetime

# 1단계 : 프롬프트 템플릿 정의
#  - 입력 할 값 : 상품명, 가격, 세일 여부
product_prompt = PromptTemplate.from_template(
    """
    다음 상품에 대한 매력적인 설명을 작성해주세요:

    사용자명 : {user_name}
    상품명 : {product_name}
    정가 : {price}원
    할인율: {discount_rate}
    현재 시간: {current_time}

    -> 위 정보를 바탕으로 고객에게 어필할 수 있는 설명을 작성해주세요. 비회원인 경우 회원 가입 유도 문구를 포함해주세요.
    """
)

# 2단계 : RunnablePassthrough.assign() 으로 **실행 시점**에 값을 동적으로 추가.
#  assign : 원본 딕셔너리(사용자가 넣은 값)의 모든 키는 유지하면서 새로운 키만 추가/덮어쓰기
#  이 때 lambda 함수를 사용한다.

chain = (
    RunnablePassthrough.assign(
        discount_rate=lambda x : 30 if x.get('is_on_sale') else 0,
        current_time=lambda x : datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분"),
        user_name=lambda x : x.get("user_name", None)
    )
    | product_prompt
    | model
    | StrOutputParser()
)

# product_name, price, is_on_sale
result = chain.invoke({
    "user_name": "소미노",
    "product_name": "무선 블루투스 이어폰",
    "price": 100000,
    "is_on_sale": True
})

print(result)

🌟 소미노의 무선 블루투스 이어폰 🌟

소리를 입다! 🎧

당신의 일상에 편안함과 스타일을 더해줄 소미노의 무선 블루투스 이어폰을 소개합니다. 정가 100,000원이지만, 지금 특별 할인된 가격 70,000원으로 만나보실 수 있습니다! (30% 할인) 

이 이어폰은 뛰어난 음질과 긴 배터리 수명을 자랑하며, 어떤 환경에서도 뚜렷한 사운드를 경험할 수 있도록 디자인되었습니다. 가벼운 무게와 인체공학적 디자인으로 하루 종일 착용해도 부담이 없으며, BT 5.0 기술로 끊김 없는 안정적인 연결이 가능합니다. 

특별한 순간을 놓치지 마세요. 🎶 지금 구매하시면, 핸즈프리 통화와 음악 감상을 더욱 편리하게 즐기실 수 있습니다. 운동, 출퇴근, 또는 여유로운 카페 시간에 이 이어폰과 함께 완벽한 음질을 사로잡아 보세요.

비회원이신가요? 지금 회원 가입을 하시면 추가 할인 혜택과 다양한 이벤트 소식도 받아보실 수 있습니다! 

행동할 시간입니다. **소미노 무선 블루투스 이어폰**으로 당신의 소리 세계를 확장해보세요! 🔊✨

[지금 구매하기]


## 2. RunnableParallel

`RunnablePassthrough`로 입력을 제어하는 방법을 알아봤어요. 이제 여러 체인을 동시에 실행하는 `RunnableParallel(러너블 패러렐)`을 살펴볼게요.

`RunnableParallel`은 딕셔너리 형태로 여러 Runnable을 정의하고, 동일한 입력을 각 Runnable에 동시에 전달해요. 결과는 키-값 쌍의 딕셔너리로 반환돼요.

### 주요 특징
- 여러 LLM 호출을 동시에 실행해 총 응답 시간을 줄여요
- 각 Runnable의 결과를 키로 구분해 딕셔너리로 반환해요
- 같은 입력으로 다양한 관점의 응답을 한 번에 생성할 때 유용해요

In [10]:
from langchain_core.runnables import RunnableParallel

# ---------------------------------------------------
# RunnableParallel: 같은 입력으로 여러 체인을 동시에 실행하기
# ---------------------------------------------------
# ============================================================
# TODO: 아래 parallel_chain을 실행하고 "장점", "단점", "활용분야" 키를 확인해요
# 힌트: parallel_chain.invoke({"topic": "원하는 주제"})
# 예상 결과: {"장점": "...", "단점": "...", "활용분야": "..."} 딕셔너리가 반환돼요
# ============================================================

# 같은 topic 입력으로 3가지 관점의 응답을 동시에 생성하는 프롬프트

prompt1 = PromptTemplate.from_template("{topic}의 장점을 3줄로 나열해줘")
prompt2 = PromptTemplate.from_template("{topic}의 단점을 3줄로 나열해줘")
prompt3 = PromptTemplate.from_template("{topic}의 주요 활용 분야를 3줄로 나열해줘")

parallel_chain = RunnableParallel({
    "장점": prompt1 | model | StrOutputParser(),
    "단점": prompt2 | model | StrOutputParser(),
    "활용분야": prompt3 | model | StrOutputParser()
})

result = parallel_chain.invoke({"topic": "미드 니달리"})
print(result)

{'장점': '미드 니달리는 강력한 기동성과 높은 스킬셋을 활용해 적을 빠르게 처치할 수 있습니다. 또한, 지속적인 포킹과 강력한 스노우볼링 능력으로 게임의 흐름을 주도할 수 있습니다. 마지막으로, 적의 이동을 방해하는 우월한 암살 능력으로 팀 전투에서 중요한 역할을 수행합니다.', '단점': '1. 니달리는 스킬셋이 복잡하여 숙련도가 낮은 플레이어에게는 활용이 어려울 수 있다.\n2. 초반 라인전에서 체력이 낮아 공격을 받기 쉬워, 초반 갱이나 압박에 취약하다.\n3. 팀fight에서 물리적인 저항력이 낮아, 적의 집중 공격을 받으면 금방 사망할 위험이 있다.', '활용분야': '미드 니달리는 다음과 같은 주요 활용 분야가 있습니다.\n\n1. **강력한 포킹 능력**: 니달리는 Q 스킬(투창)을 활용해 적을 지속적으로 견제하며, 원거리에서 CC를 부여할 수 있습니다.\n2. **빠른 정글 회전**: W 스킬(고양이 형태로 변신)을 이용해 정글 캠프를 빠르게 처치하고, 타겟이 적은 상태에서 빠른 정글 회전이 가능합니다.\n3. **스노우볼링**: 초반에 상대에게 강한 압박을 가하고, 사고 및 팀 전투에서 우위를 점유하여 게임 주도권을 가져올 수 있습니다.'}


In [13]:
print(result['활용분야'])

미드 니달리는 다음과 같은 주요 활용 분야가 있습니다.

1. **강력한 포킹 능력**: 니달리는 Q 스킬(투창)을 활용해 적을 지속적으로 견제하며, 원거리에서 CC를 부여할 수 있습니다.
2. **빠른 정글 회전**: W 스킬(고양이 형태로 변신)을 이용해 정글 캠프를 빠르게 처치하고, 타겟이 적은 상태에서 빠른 정글 회전이 가능합니다.
3. **스노우볼링**: 초반에 상대에게 강한 압박을 가하고, 사고 및 팀 전투에서 우위를 점유하여 게임 주도권을 가져올 수 있습니다.


### 병렬 실행의 동작 방식 이해하기

`RunnableParallel`은 여러 체인을 병렬로 실행하지만, `invoke()` 메서드는 모든 체인이 완료될 때까지 기다려요. 즉, 사용자는 가장 느린 체인이 끝날 때까지 대기해요.

**데이터 흐름**:
```
입력 {"topic": "..."} ──┬── chain1 (장점) ──┐
                        ├── chain2 (단점) ──┼── 결과 {"장점": ..., "단점": ..., "활용분야": ...}
                        └── chain3 (활용) ──┘
```

다음 셀에서 순차 실행, 병렬 실행, 비동기 실행의 시간을 직접 비교해볼게요.

In [ ]:
import time

# ---------------------------------------------------
# 순차 실행 vs 병렬 실행 vs 비동기 실행 시간 비교하기
# ---------------------------------------------------


In [15]:
from datetime import datetime

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
)
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

# ---------------------------------------------------
# 실전 예제: Passthrough + Parallel + Lambda 조합
# 하나의 상품 입력으로 여러 채널용 마케팅 카피를 동시에 생성하기
# ---------------------------------------------------

# 1. 프롬프트 정의 ( 상세 설명, 한줄 설명, 알림톡 문구)
#   상세 설명 : 사용자명, 상품명, 정가, 할인율, 현재 시간
#   한줄 설명 : 상품명, 가격, 할인율
#   알림톡 문구 : 상품명, 가격, 할인율, 현재 시간, 사용자명
# 1단계: 용도별 프롬프트 정의
# 상세 설명 — 웹 상품 페이지용
detail_prompt = PromptTemplate.from_template(
    "다음 상품에 대한 매력적인 설명을 작성해주세요:\n"
    "사용자명: {user_name}\n"
    "상품명: {product_name}\n"
    "정가: {price}원\n"
    "할인율: {discount_rate}%\n"
    "현재 시간: {current_time}\n"
    "→ 위 정보를 바탕으로 고객에게 어필할 수 있는 상세 설명을 작성해주세요. "
    "비회원(사용자명이 없거나 None인 경우)인 경우 회원 가입 유도 문구를 포함해주세요."
)

# 한 줄 요약 — 배너/광고 카피용
summary_prompt = PromptTemplate.from_template(
    "다음 상품 정보를 기반으로, 마케팅용 한 줄 카피만 1줄로 작성해주세요.\n"
    "상품명: {product_name}\n"
    "정가: {price}원\n"
    "할인율: {discount_rate}%\n"
    "→ 말 줄임표 없이 완전한 문장으로."
)

# 알림톡 문구 — 카카오 알림톡/문자 발송용
sms_prompt = PromptTemplate.from_template(
    "아래 정보를 이용해, 카카오 알림톡 형식의 짧은 문구를 작성해주세요.\n"
    "상품명: {product_name}\n"
    "정가: {price}원\n"
    "할인율: {discount_rate}%\n"
    "현재 시간: {current_time}\n"
    "사용자명: {user_name}\n"
    "조건:\n"
    "- 회원이면 이름을 직접 불러주는 느낌으로 시작\n"
    "- 비회원이면 '고객님'으로 부르고, 마지막에 회원가입 유도 문구 1줄 추가\n"
)

# 2단계 : RunnablePassthrough.assign()을 이용해서 원본 입력에 파생 값 자동 주입
#  입력 : {product_name, price, is_on_sale}
#  출력 : {product_name, price, is_on_sale, discount_rate, current_time, user_name}
enrich_chain = RunnablePassthrough.assign(
    discount_rate=lambda x : 30 if x.get('is_on_sale') else 0,
    current_time=lambda x : datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분"),
    user_name=lambda x : x.get("user_name", None)
)

# 3단계 : RunnableParallel - enrich 된 입력으로 4개의 작업을 병렬 실행.
parallel_chain = RunnableParallel(
    원본입력=RunnablePassthrough(), # 확장된 입력을 보존
    상세설명=detail_prompt | model | StrOutputParser(),
    한줄요약=summary_prompt | model | StrOutputParser(),
    알림톡문구=sms_prompt | model | StrOutputParser()
)

# 4단계 : RunnableLambda - 병렬 처리 결과를 하나의 딕셔너리로 합치기
def merge_parallel_outputs(data):
    original = data['원본입력'] # 딕셔너리로 되어 있음.
    return {
        **original, # **{'a': 10, 'b': 20} => a=10, b=20
        "상세설명": data['상세설명'],
        "한줄요약": data['한줄요약'],
        "알림톡문구": data['알림톡문구']
    }

merge_chain = RunnableLambda(merge_parallel_outputs)

# 5단계 
chain = (
    enrich_chain | parallel_chain | merge_chain
)

result = chain.invoke({
    "product_name": "무선 블루투스 이어폰",
    "price": 100000,
    "is_on_sale": True
})

result

{'product_name': '무선 블루투스 이어폰',
 'price': 100000,
 'is_on_sale': True,
 'discount_rate': 30,
 'current_time': '2026년 03월 23일 16시 13분',
 'user_name': None,
 '상세설명': '### ✨ 무선 블루투스 이어폰 – 완벽한 음악 경험을 지금 누리세요! 🎶\n\n음악을 사랑하는 여러분, 주목하세요! 무선 블루투스 이어폰이 특별 할인된 가격에 출시되었습니다. 정가는 100,000원이지만, 지금은 **30% 할인된 70,000원**에 만나보실 수 있습니다! 🎉\n\n**제품 특징:**\n\n- **무선 편안함**: 복잡한 선은 잊어버리세요! 무선 블루투스 기술로 언제 어디서나 자유롭게 음악을 즐길 수 있습니다.\n- **탁월한 사운드**: 깊고 풍부한 베이스와 깨끗한 고음을 자랑하는 이어폰으로, 음악의 모든 순간을 생생하게 경험하세요.\n- **편리한 통화 기능**: 내장 마이크가 탑재되어 있어, 전화 통화도 편리하게! 다양한 스트리밍 서비스와의 호환성으로 영화와 게임도 문제 없이 즐길 수 있습니다.\n- **오랜 배터리 수명**: 한 번 충전으로 긴 시간 사용 가능한 배터리로, 하루 종일 끊김 없이 음악을 감상하세요.\n\n구매 시 **무선 블루투스 이어폰을 더욱 특별하게 사용할 수 있는 팁과 정보를 제공받고 싶으신가요?** 비회원이신 지금, 회원 가입을 통해 더욱 다양한 혜택과 이벤트 소식을 직접 받아보세요! 🎁\n\n**지금 바로 구매하시고, 음악과 함께하는 새로운 라이프스타일을 경험해보세요!** \n\n이 기회를 놓치지 마세요! 재고 소진이 빠를 수 있으니 서둘러주세요! ⏰',
 '한줄요약': '30% 할인된 가격으로 자유로운 음악 감상을 경험할 수 있는 무선 블루투스 이어폰을 만나보세요!',
 '알림톡문구': '고객님, 지금 무선 블루투스 이어폰을 정가 100,000원에서 30% 할인된 가격으로 만나보세요! 🛒✨ 한정 수량이니 서두르세요! 더 

### 실전 응용: Passthrough + Parallel + Lambda 조합

세 가지 Runnable을 조합해 **하나의 입력으로 여러 형태의 결과물을 동시에 생성**하는 파이프라인을 만들어볼게요.

위 코드 셀의 파이프라인 구조를 정리하면 다음과 같아요:

```
원본 입력 → Passthrough.assign() → RunnableParallel → Lambda(합치기)
            (할인율/시간 추가)      ├─ 상세설명 (LLM)
                                   ├─ 한줄요약 (LLM)
                                   ├─ 알림톡문구 (LLM)
                                   └─ 원본입력 (Passthrough)
```

이 패턴은 실무에서 자주 활용돼요:
- 같은 데이터로 다양한 채널용 콘텐츠를 동시에 생성할 때 (웹, 앱 푸시, 문자)
- A/B 테스트용 여러 버전의 카피를 한 번에 생성할 때
- 분석 결과를 여러 관점에서 동시에 요약할 때

## 3. RunnableLambda

`RunnableParallel`로 병렬 실행을 다뤄봤어요. 이번에는 일반 Python 함수를 체인에 삽입하는 `RunnableLambda(러너블 람다)`를 살펴볼게요.

`RunnableLambda`는 일반 Python 함수를 Runnable로 변환해 파이프라인에 포함시켜요. 체인 중간에 커스텀 로직을 삽입할 수 있어요.

### 주요 활용 사례
- 입력 전처리: 사용자 친화적 형식을 LLM 입력 형식으로 변환해요
- 출력 후처리: LLM 응답에서 특정 부분만 추출하거나 포맷팅해요
- 데이터 타입 변환: 문자열 → 딕셔너리, 딕셔너리 → 문자열 등의 변환이 필요할 때 사용해요

In [18]:
from langchain_core.runnables import RunnableLambda

# ---------------------------------------------------
# RunnableLambda: 일반 Python 함수를 파이프라인에 삽입하기
# ---------------------------------------------------

# AI가 응답을 할건데, 응답 앞에 [AI 응답] 형식을 붙여주기.
def add_prefix(text):
    return f"[AI 응답] {text}"

prompt = PromptTemplate.from_template("{question}에 대해 간단히 설명해줘.")

chain = (
    prompt | model | StrOutputParser() | RunnableLambda(add_prefix)
)

result = chain.invoke({"question" : "딥러닝이란?"})
print(result)

[AI 응답] 딥러닝은 인공지능의 한 분야로, 인공 신경망을 기반으로 한 기계 학습 기술입니다. 이 기술은 인간의 뇌가 정보를 처리하는 방식에서 영감을 받아 설계되었습니다. 딥러닝은 여러 층의 신경망(다층 신경망)을 사용하여 데이터에서 특징을 자동으로 학습하고, 이미지 인식, 자연어 처리, 음성 인식 등 다양한 분야에서 뛰어난 성능을 보입니다.

딥러닝의 주요 특징은 대량의 데이터와 강력한 컴퓨팅 자원을 활용하여 모델을 학습시킬 수 있다는 점입니다. 이를 통해 복잡한 패턴이나 구조를 잡아내고, 예측 및 분류 작업을 수행하는 데 매우 효과적입니다.


In [19]:
# ---------------------------------------------------
# 여러 RunnableLambda를 순서대로 연결하기 (데이터 타입 변환 포함)
# ---------------------------------------------------

# RunnableLambda로 감싼 함수의 인자는 항상 **이전 단계의 전체 출력**을 받는다.
#    이전 단계가 딕셔너리로 반환하면, 함수도 딕셔너리를 인자로 받아야 한다.

def extract_keywords(text):
    """미리 정의된 키워드 목록과 매칭해 추출하기"""
    keywords = ["파이썬", "머신러닝", "딥러닝", "데이터", "AI", "알고리즘"]
    found = [kw for kw in keywords if kw in text]
    # 반환 타입이 문자열 → 딕셔너리로 바뀜 (다음 단계에서 딕셔너리로 받음)
    return {"text": text, "keywords": found, "keyword_count": len(found)}

def format_output(data):
    """딕셔너리를 사람이 읽기 좋은 형태로 포맷팅 — 딕셔너리 → 문자열 변환"""
    return f"""
원본 텍스트: {data['text'][:100]}...

추출된 키워드: {', '.join(data['keywords']) if data['keywords'] else '없음'}
키워드 개수: {data['keyword_count']}
"""

prompt = PromptTemplate.from_template("{topic}에 대해 설명해줘.")

chain = (
    prompt | model | StrOutputParser() # 모델 출력 -> str
    | RunnableLambda(extract_keywords) # str -> dict 변환
    | RunnableLambda(format_output) # dict -> str 변환
)

result = chain.invoke({"topic": "파이썬 프로그래밍"})
print(result)


원본 텍스트: 파이썬(Python)은 고급 프로그래밍 언어로, 1991년 귀도 반 로섬(Guido van Rossum)에 의해 처음 개발되었습니다. 파이썬은 코드가 간결하고 읽기 쉬운 특성을 가...

추출된 키워드: 파이썬, 머신러닝, 데이터, AI
키워드 개수: 4



In [20]:
# ---------------------------------------------------
# RunnableLambda를 입력 전처리·출력 후처리 양쪽에 사용하기
# ---------------------------------------------------
# ============================================================
# TODO: language를 "영어"로 바꿔서 실행하고 응답이 달라지는지 확인해요
# 힌트: invoke()의 딕셔너리에서 "language" 값을 변경해요
# 예상 결과: prepare_input()이 "{language}로 {topic}에 대해 설명해줘" 문자열을 생성해요
# ============================================================

# 사용자의 입력을 전처리 하는 함수
def prepare_input(data):
    # {topic, language} -> {question} 딕셔너리로 변환
    topic = data.get("topic", "")
    language = data.get("language", "한국어")

    return {
        "question": f"{language}로 {topic}에 대해 설명해줘"
    }

def process_output(text):
    return f"[🤖 AI 응답 🤖] {text}"

chain = (
    RunnableLambda(prepare_input) # {topic, langauge} -> {question}
    | PromptTemplate.from_template("{question}")
    | model
    | StrOutputParser()
    | RunnableLambda(process_output) # 모델의 응답에 AI 응답 추가.
)

result = chain.invoke({
    "topic": "머신러닝",
    "language": "일본어"
})

print(result)

[🤖 AI 응답 🤖] 機械学習（きかいがくしゅう、Machine Learning）は、コンピュータがデータを用いて自ら学び、経験を通じて性能を向上させるためのアルゴリズムや技術の集合を指します。これは、明示的にプログラムされることなく、パターンを認識したり予測を行ったりする能力を持つシステムを作成することを目的としています。

機械学習は主に以下の3つのカテゴリに分けられます：

1. **教師あり学習（きょうしありがくしゅう）**: 教師データ（入力とそれに対応する正しい出力）が与えられ、それに基づいて学習する方法です。例えば、スパムメールを分類するために、スパムと正常なメールの例を使ってモデルを訓練します。

2. **教師なし学習（きょうしなしがくしゅう）**: 教師データがない場合に使用される方法で、データの中からパターンや構造を見つけることを目的とします。クラスタリング（データをグループに分ける）などがこのカテゴリに含まれます。

3. **強化学習（きょうせいがくしゅう）**: エージェントが環境と相互作用し、試行錯誤を通じて最適な行動を学習する方法です。報酬を最大化するための戦略を学ぶことが求められます。

機械学習は、ビジネス、医療、金融、製造業など、さまざまな分野で利用されており、画像認識、音声認識、自然言語処理などの応用も広がっています。


## 4. 종합 예제: 세 가지 Runnable 조합하기

`RunnablePassthrough`, `RunnableParallel`, `RunnableLambda`를 모두 배웠어요. 이번에는 세 가지를 함께 조합해 실무형 마케팅 패키지 생성 파이프라인을 만들어볼게요.

이 예제는 앞서 섹션 2에서 다룬 실전 응용과 동일한 구조이지만, 최종 출력을 딕셔너리 대신 사람이 바로 읽을 수 있는 보고서 형태의 문자열로 반환해요.

In [ ]:
from datetime import datetime

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda,
)
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

# ---------------------------------------------------
# 종합 예제: 세 가지 Runnable 조합 (보고서 형태 출력)
# ---------------------------------------------------
# 섹션 2의 실전 예제와 동일한 파이프라인 구조이지만,
# Lambda 최종 단계에서 딕셔너리 대신 보고서 문자열로 반환해요

# 1단계: 프롬프트 정의
detail_prompt = PromptTemplate.from_template(
    "다음 상품에 대한 매력적인 설명을 작성해주세요:\n"
    "사용자명: {user_name}\n"
    "상품명: {product_name}\n"
    "정가: {price}원\n"
    "할인율: {discount_rate}%\n"
    "현재 시간: {current_time}\n"
    "→ 위 정보를 바탕으로 고객에게 어필할 수 있는 상세 설명을 작성해주세요. "
    "비회원(사용자명이 없거나 None인 경우)인 경우 회원 가입 유도 문구를 포함해주세요."
)

summary_prompt = PromptTemplate.from_template(
    "다음 정보를 바탕으로 마케팅용 한 줄 카피를 1줄만 작성해주세요.\n"
    "상품명: {product_name}\n"
    "정가: {price}원\n"
    "할인율: {discount_rate}%\n"
    "→ 말 줄임표 없이 완전한 문장으로."
)

sms_prompt = PromptTemplate.from_template(
    "아래 정보를 이용해 카카오 알림톡 형식의 짧은 문구를 작성해주세요.\n"
    "상품명: {product_name}\n"
    "정가: {price}원\n"
    "할인율: {discount_rate}%\n"
    "현재 시간: {current_time}\n"
    "사용자명: {user_name}\n"
    "조건:\n"
    "- 회원이면 이름을 직접 불러주는 느낌으로 시작\n"
    "- 비회원이면 '고객님'으로 부르고, 마지막에 회원가입 유도 문구 1줄 추가\n"
)

# 2단계: assign() — 파생 값 자동 주입
enrich_chain = RunnablePassthrough.assign(
    discount_rate=lambda x: 30 if x.get("is_on_sale") else 0,
    current_time=lambda x: datetime.now().strftime("%Y년 %m월 %d일 %H시 %M분"),
    user_name=lambda x: x.get("user_name", None),
)

# 3단계: parallel() — 3개 LLM 동시 호출
parallel_chain = RunnableParallel(
    원본입력=RunnablePassthrough(),
    상세설명=detail_prompt | model | StrOutputParser(),
    한줄요약=summary_prompt | model | StrOutputParser(),
    알림톡문구=sms_prompt | model | StrOutputParser(),
)

# 4단계: Lambda — 최종 보고서 문자열로 합치기
# 섹션 2와 달리 딕셔너리가 아닌 사람이 바로 읽을 수 있는 문자열 형태로 출력해요
def format_marketing_package(data: dict) -> str:
    """병렬 결과를 보고서 형태의 단일 문자열로 합치기"""
    original = data["원본입력"]
    user_name = original.get("user_name")
    user_label = user_name if user_name else "비회원(게스트)"

    header = (
        "=== 상품 마케팅 패키지 ===\n"
        f"- 사용자: {user_label}\n"
        f"- 상품명: {original['product_name']}\n"
        f"- 정가: {original['price']}원\n"
        f"- 할인율: {original['discount_rate']}%\n"
        f"- 기준 시간: {original['current_time']}\n"
    )

    detail = f"\n[상세 설명]\n{data['상세설명']}\n"
    summary = f"\n[한 줄 요약]\n{data['한줄요약']}\n"
    sms = f"\n[알림톡 문구]\n{data['알림톡문구']}\n"

    return header + detail + summary + sms

format_chain = RunnableLambda(format_marketing_package)

# 전체 체인: enrich → parallel → format
combined_chain = enrich_chain | parallel_chain | format_chain

# 5단계: 실행
result = combined_chain.invoke({
    # "user_name": "미노님",  # 주석을 풀면 회원 케이스
    "product_name": "무선 블루투스 이어폰",
    "price": 99000,
    "is_on_sale": True
})

print(result)

---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

| Runnable | 역할 | 주요 사용법 |
|----------|------|-------------|
| `RunnablePassthrough` | 입력을 그대로 통과시켜요 | 분기점에서 원본 데이터 유지 |
| `RunnablePassthrough.assign()` | 원본 유지 + 새 키 추가 | 파생 값(할인율, 타임스탬프) 동적 주입 |
| `RunnableParallel` | 여러 체인 동시 실행 | 같은 입력으로 다양한 출력 생성 |
| `RunnableLambda` | Python 함수를 Runnable로 변환 | 전처리·후처리·데이터 타입 변환 |

세 가지를 조합하면 `enrich → parallel → merge` 패턴으로 복잡한 실무 파이프라인을 선언적으로 표현할 수 있어요.

## 다음 노트북 예고

다음 노트북에서는 **메모리(Memory)**와 **대화 이력 관리**를 다뤄요. 지금까지 수동으로 관리하던 대화 히스토리를 LangChain 컴포넌트가 자동으로 처리하는 방법을 알아볼게요.